# Standardization Practice — `trim` + `upper` (country)

Part 3 of the interview drill on the dirty retail-orders dataset. Picks up exactly where **02 Null Handling Practice** left off: `orders_clean` — 20 deduplicated rows with `quantity_clean` (INT) added. The remaining dirt: `$`-prefixed prices, mixed country casing, and a future order date.

**The question:** `country` is a STRING containing `' canada'`, `'USA'`, `'usa'`, `' USA '`, `'CANADA'`, `' Canada'`. Add a NEW column `country_clean` (don't overwrite `country`) so every country converges to one consistent form — exactly 3 distinct values expected: `USA`, `UK`, `CANADA`.

Dedup logic lives in Notebook 01, null handling in Notebook 02, and the full dirty-data menu in `demo_notebook` (Cell 4).

In [0]:
orders_clean_rows = [
    ("ORD1001", "John Smith", "USA", "Laptop", "2", "$1200.00", "2024-05-13", "2024-06-02 08:15:00", 2),
    ("ORD1002", " jane doe ", " canada", "Mouse", "5", "$25.50", "2024-05-14", "2024-06-02 08:15:00", 5),
    ("ORD1003", None, "USA", "Keyboard", "1", "$45.00", "2024-05-15", "2024-06-02 08:15:00", 1),
    ("ORD1004", "Alice Johnson", "UK", "Monitor", "-5", "$180.00", "2024-05-16", "2024-06-02 08:15:00", 1),
    ("ORD1005", "bob brown", "usa", "Laptop", "3", None, "2024-05-17", "2024-06-02 08:15:00", 3),
    ("ORD1006", "Charlie Davis", "Canada", "Headphones", "0", "$60.00", "2024-05-18", "2024-06-02 08:15:00", 0),
    ("ORD1007", None, "USA", "Mouse", "9", "$25.50", "2024-05-22", "2024-06-03 10:05:00", 9),
    ("ORD1008", "David Lee", " USA ", "Keyboard", "1", "$45.00", "2024-05-20", "2024-06-02 08:15:00", 1),
    ("ORD1009", "Eve Wilson", "usa", "Monitor", "2", "$180.00", "2024-06-10", "2024-06-03 09:40:00", 2),
    ("ORD1010", "Frank Miller", "Canada", "Laptop", "1", "$1200.00", "2050-01-01", "2024-06-02 08:15:00", 1),
    ("ORD1011", " grace taylor ", "USA", "Mouse", None, "$25.50", "2024-05-23", "2024-06-02 08:15:00", 1),
    ("ORD1012", "Henry Adams", "UK", "Headphones", "2", "$0.00", "2024-05-24", "2024-06-02 08:15:00", 2),
    ("ORD1013", "Ivy Clark", "CANADA", "Monitor", "2", "$180.00", "2024-05-25", "2024-06-02 08:15:00", 2),
    ("ORD1014", None, "usa", "Keyboard", "3", "$45.00", "2024-05-26", "2024-06-02 08:15:00", 3),
    ("ORD1015", "Jack White", "USA", "Laptop", None, None, None, "2024-06-02 08:15:00", 1),
    ("ORD1016", "Karen Hall", " Canada", "Mouse", "5", "$25.50", "2024-05-28", "2024-06-02 08:15:00", 5),
    ("ORD1017", "Leo King", "usa", "Monitor", "-2", "$180.00", "2024-05-29", "2024-06-02 08:15:00", 1),
    ("ORD1018", "Mona Scott", "USA", "Headphones", "7", "$60.00", "2024-05-30", "2024-06-03 11:20:00", 7),
    ("ORD1019", "Nick Green", "UK", "Keyboard", " 2 ", "$45.00", "2024-05-31", "2024-06-02 08:15:00", 2),
    ("ORD1020", "Olivia Baker", "Canada", "Mouse", "3", "$25.50", "2024-06-01", "2024-06-02 08:15:00", 3),
]

orders_clean = spark.createDataFrame(
    orders_clean_rows,
    "order_id STRING, customer_name STRING, country STRING, product STRING, quantity STRING, unit_price STRING, order_date STRING, ingested_at STRING, quantity_clean INT",
)

display(orders_clean)

## 1. Country standardization — notes

**Question:** on `orders_clean`, add a new column `country_clean` where every variant of the same country maps to one value. Exactly 3 distinct values expected: `USA`, `UK`, `CANADA`.

**Which form to standardize to — and why:**

* `upper(trim(...))` — the standard for **join keys and matching**. Silver/gold layers standardize to a canonical match form: `usa` and `USA` must become the same entity before any GROUP BY or join.
* `initcap` — for the **presentation layer** (reports, dashboards humans read), never for keys. It also mangles abbreviations: `'UK'` → `'Uk'`, `'usa'` → `'Usa'`.

**Why standardize BEFORE aggregating revenue per country:** `GROUP BY country` on dirty strings puts `usa` and `USA` in separate groups — one country's revenue splits into fragments, and it fails **silently**: no error, just wrong numbers in the report.

**When a mapping/dimension table beats string functions:** string functions fix only **mechanical** variation (casing, whitespace). A mapping/dim table wins when variation is **semantic** — `"U.S."`, `"U.S.A."`, `"United States"` — no string function maps those together. Also: many-to-one mappings, business can change mappings without code deploys, governance/audit (who changed what, when). Rule of thumb: **string functions for formatting, mapping tables for meaning**.

In [0]:
from pyspark.sql.functions import col, trim, upper

orders_std = orders_clean.withColumn("country_clean", upper(trim(col("country"))))

orders_std.display()

In [0]:
orders_std.select("order_id", "country", "country_clean") \
    .filter(col("order_id").isin("ORD1002", "ORD1005", "ORD1008", "ORD1013", "ORD1016")) \
    .display()

display(orders_std.select("country_clean").distinct())

## Key Concepts Summary

| Concept | Rule / Distinction |
| --- | --- |
| `trim` + `upper` | mechanical standardization — fixes casing and whitespace only |
| UPPER vs initcap | UPPER for join keys / matching (silver layer); initcap for presentation only; initcap mangles `'UK'` → `'Uk'` |
| New column vs overwrite | `country_clean` added, raw `country` kept for audit — same pattern as `quantity_clean` |
| Standardize before aggregate | dirty strings split one country into multiple GROUP BY groups — silent wrong revenue |
| Mapping/dim table | beats string functions when variation is semantic (`"U.S."`, `"United States"`), many-to-one, or changes without code deploys |
| Assign the result | `.display()` alone stores nothing — assign to a stage variable (`orders_std`) so the next stage can build on it |

**Lessons from my attempts:**

* `initcap` also converges the variants but produces `Usa` / `Uk` / `Canada` — fails the acceptance form (USA, UK, CANADA) and mangles abbreviations.
* First attempt ended in `.display()` without assigning — the kernel kept no `country_clean`, so the next stage had nothing to build on.
* `UPPER` was the right choice; the missing interview point was justifying **why** (join keys / canonical match form), not only naming the function.